# Figure 2

**Cellpose models trained through Mycol support rapid generation of CFU-counting models**

- **Panels** — **a** workflow strip (seeded geometry) · **b-d** plate photographs · **e-g** true vs
  predicted CFU count
- **Needs** — the three `case_study_1/` sessions; `assets/` for the S. aureus and mCount photographs
  and `cellpose_count_evaluation.csv` (panel **g**, from `analyses/cellpose_mcount_evaluation.ipynb`)
- **Writes** — `output/Figure_2.svg` and its 300 dpi `.png`, the raster the manuscript embeds;
  intermediates in `output/panels/`
- **Kernel** — `mycol_colonies_env`, run top to bottom


In [ ]:
import re
import subprocess
from pathlib import Path

from PIL import Image

ASSETS = Path("assets")
OUTPUT = Path("output")
PANELS = OUTPUT / "panels"          # intermediates: the workflow strip and the data panels
REPO_ROOT = Path.cwd().parents[2]
PANELS.mkdir(parents=True, exist_ok=True)


def wrote(path):
    print(f"  wrote {path.name:38} {path.stat().st_size:>9,} B")


def rasterise(svg_path, png_path, dpi=300):
    """Render the finished SVG to PNG at `dpi` with the Chrome kaleido installs.

    Chrome is already a dependency - plotly's static export drives it. SVG user units
    are CSS px at 96 dpi, hence the device scale factor of dpi/96.
    """
    try:
        from choreographer.cli import _cli_utils
        chrome = str(_cli_utils.get_chrome_sync())
    except Exception:
        chrome = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"
        if not Path(chrome).exists():
            raise RuntimeError("no Chrome - run `plotly_get_chrome`, or install Google Chrome")

    svg = svg_path.read_text()
    w, h = (round(float(v)) for v in
            re.search(r'viewBox="0 0 ([\d.]+) ([\d.]+)"', svg[:800]).groups())
    page = OUTPUT / "_render.html"
    page.write_text("<!doctype html><meta charset=utf-8><style>"
                    "html,body{margin:0;padding:0;background:#fff}svg{display:block}</style>"
                    + svg[svg.index("<svg"):])
    # Chrome reserves ~87 CSS px of window furniture even headless and clips the page
    # by that much, so render into a taller window and crop back.
    scale, raw = dpi / 96, OUTPUT / "_render.png"
    subprocess.run([chrome, "--headless", "--disable-gpu", "--hide-scrollbars",
                    f"--force-device-scale-factor={scale}", f"--window-size={w},{h + 200}",
                    f"--screenshot={raw.resolve()}", page.resolve().as_uri()],
                   check=True, capture_output=True)
    with Image.open(raw) as im:
        im.crop((0, 0, round(w * scale), round(h * scale))).save(png_path)
    raw.unlink()
    page.unlink()
    wrote(png_path)


## Panel a - the workflow strip

Four step panels plus the flow that composes them, all from seeded geometry, so the strip is
deterministic. The assembly cell inlines `figure2_flow.svg`.


In [ ]:
import math
import random

PREFIX = "figure2"
PANEL_W, PANEL_H = 240, 250
ART_SHIFT = -28                     # art sits high in the panel, title below it
TITLE_Y = 224                       # title baseline
GAP, PAD, STRIP_MARGIN = 52, 28, 16

STRIP_STYLE = """
    text     { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
               Helvetica, Arial, sans-serif; fill:#0f172a; }
    .title   { font-size:13px; font-weight:600; text-anchor:middle; }

    .panel   { fill:#ffffff; stroke:#cbd5e1; stroke-width:1.5; }
    .agar    { fill:#f4efdd; stroke:#0f172a; stroke-width:1.2; }
    .dish    { fill:#ffffff; stroke:#0f172a; stroke-width:1.6; }

    .cell    { fill:#94a3b8; fill-opacity:0.30; stroke:#0f172a; stroke-width:0.9; }
    .mask    { fill:#e5431e; fill-opacity:0.17; stroke:#0f172a; stroke-width:2;
               stroke-linejoin:round; }
    .cursor  { fill:#0f172a; stroke:#ffffff; stroke-width:1.1; stroke-linejoin:round; }

    .err     { fill:none; stroke:#c2410c; stroke-width:1.6; stroke-dasharray:4 3.5; }
    .card    { fill:#ffffff; stroke:#0f172a; stroke-width:1.2; }
    .glyph   { fill:#94a3b8; fill-opacity:0.45; stroke:#0f172a; stroke-width:0.9; }
    .barA    { fill:#5289C7; fill-opacity:0.75; }
    .dl      { fill:none; stroke:#1d4ed8; stroke-width:3; stroke-linecap:round;
               stroke-linejoin:round; }
    .arrow   { fill:none; stroke:#334155; stroke-width:1.8; }
    .arrowh  { fill:#334155; }
"""

CURSOR = ('<path class="cursor" transform="translate({x},{y}) scale({s})" '
          'd="M 0,0 0,15 3.9,11.4 6.7,17 9.3,15.7 6.5,10.3 11.9,10.1 Z" />')


def harrow(x1, x2, y):
    return (f'<path class="arrow" d="M {x1},{y} H {x2 - 7}" />'
            f'<path class="arrowh" d="M {x2},{y} {x2 - 8},{y - 4.8} {x2 - 8},{y + 4.8} Z" />')


# ── the plate: 24 colonies scattered on the agar, no two touching, plus one
#    deliberately touching pair and one colony the segmenter will miss ──
PLATE_CX, PLATE_CY, PLATE_R = 120, 138, 84
_rng = random.Random(7)

_points, _guard = [], 0
while len(_points) < 24 and _guard < 8000:          # rejection sampling, min 17 apart
    _guard += 1
    a = _rng.uniform(0, 2 * math.pi)
    r = (PLATE_R - 21) * math.sqrt(_rng.random())
    p = (PLATE_CX + r * math.cos(a), PLATE_CY + r * math.sin(a))
    if all(math.dist(p, q) >= 17 for q in _points):
        _points.append(p)

COLONIES = [(x, y, _rng.uniform(5.0, 7.0)) for x, y in _points]

_central = sorted(range(len(COLONIES)),
                  key=lambda i: math.dist(COLONIES[i][:2], (PLATE_CX, PLATE_CY)))
PAIR, MISSED = _central[0], _central[5]
COLONIES.append((COLONIES[PAIR][0] + 9.5, COLONIES[PAIR][1] + 4.5, 6.0))   # touching pair
TOUCH = len(COLONIES) - 1


def plate(draw):
    """The dish plus its colonies, each drawn by the given callable."""
    out = [f'<circle class="dish" cx="{PLATE_CX}" cy="{PLATE_CY}" r="{PLATE_R}" />',
           f'<circle class="agar" cx="{PLATE_CX}" cy="{PLATE_CY}" r="{PLATE_R - 8}" />']
    out += [draw(i, x, y, r) for i, (x, y, r) in enumerate(COLONIES)]
    return "\n  ".join(out)


def step1_plate():
    return plate(lambda i, x, y, r: f'<circle class="cell" cx="{x:.1f}" cy="{y:.1f}" r="{r:.1f}" />')


def step2_segment():
    """Cellpose masks: the touching pair is merged, one colony is missed."""
    a, b = COLONIES[PAIR], COLONIES[TOUCH]
    mx, my = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
    ang = math.degrees(math.atan2(b[1] - a[1], b[0] - a[0]))

    def draw(i, x, y, r):
        base = f'<circle class="cell" cx="{x:.1f}" cy="{y:.1f}" r="{r:.1f}" />'
        if i in (MISSED, PAIR, TOUCH):
            return base
        return base + f'<circle class="mask" cx="{x:.1f}" cy="{y:.1f}" r="{r + 1:.1f}" />'

    return plate(draw) + "\n  " + (
        f'<ellipse class="mask" cx="{mx:.1f}" cy="{my:.1f}" rx="13.5" ry="7.6" '
        f'transform="rotate({ang:.1f},{mx:.1f},{my:.1f})" />')


def step3_curate():
    """The pair split, the missed colony picked up, both fixes flagged."""
    a, b, missed = COLONIES[PAIR], COLONIES[TOUCH], COLONIES[MISSED]
    mx, my = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
    body = plate(lambda i, x, y, r:
                 f'<circle class="cell" cx="{x:.1f}" cy="{y:.1f}" r="{r:.1f}" />'
                 f'<circle class="mask" cx="{x:.1f}" cy="{y:.1f}" r="{r + 1:.1f}" />')
    return (body + "\n  "
            + f'<ellipse class="err" cx="{mx:.1f}" cy="{my:.1f}" rx="21" ry="15" />'
            + f'<circle class="err" cx="{missed[0]:.1f}" cy="{missed[1]:.1f}" r="14" />'
            + "\n  " + CURSOR.format(x=mx + 12, y=my + 6, s=1.0))


def step4_export():
    """Per-plate counts leaving as a file."""
    out = ['<rect class="card" x="46" y="62" width="148" height="104" rx="8" />']
    for i, w in enumerate([58, 44, 68, 36]):
        y = 84 + i * 22
        out.append(f'<circle class="glyph" cx="66" cy="{y}" r="7.5" />')
        out.append(f'<rect class="barA" x="82" y="{y - 5}" width="{w}" height="10" rx="5" />')
    out.append('<path class="dl" d="M 120,180 V 200" />')
    out.append('<path d="M 120,212 111,199 129,199 Z" fill="#1d4ed8" />')
    out.append('<path class="dl" d="M 96,214 V 222 H 144 V 214" />')
    return "\n  ".join(out)


STEPS = [("step1_plate", step1_plate, "Agar plate with colonies"),
         ("step2_segment", step2_segment, "Cellpose segmentation"),
         ("step3_curate", step3_curate, "Manual correction"),
         ("step4_export", step4_export, "Export CFU counts")]

STEP_SVG = """<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<svg width="{w}" height="{h}" viewBox="0 0 {w} {h}" version="1.1"
     xmlns="http://www.w3.org/2000/svg">
  <title>{title}</title>
  <style>{style}  </style>
  <rect class="panel" x="0.75" y="0.75" width="{iw}" height="{ih}" rx="10" />
  <g transform="translate(0,{shift})">
  {art}
  </g>
  <text class="title" x="{tx}" y="{ty}">{title}</text>
</svg>
"""

# one standalone SVG per step, and the same bodies composed into the flow
bodies = []
for name, fn, title in STEPS:
    svg = STEP_SVG.format(w=PANEL_W, h=PANEL_H, iw=PANEL_W - 1.5, ih=PANEL_H - 1.5,
                          style=STRIP_STYLE, art=fn(), title=title,
                          tx=PANEL_W / 2, ty=TITLE_Y, shift=ART_SHIFT)
    (PANELS / f"{PREFIX}_{name}.svg").write_text(svg)
    inner = re.sub(r"^.*?<svg[^>]*>", "", svg, flags=re.S).rsplit("</svg>", 1)[0]
    bodies.append(re.sub(r"<style>.*?</style>", "", inner, flags=re.S).strip())

n = len(STEPS)
band_w = PAD * 2 + n * PANEL_W + (n - 1) * GAP
W = STRIP_MARGIN * 2 + band_w
H = STRIP_MARGIN * 2 + PAD * 2 + PANEL_H
y = STRIP_MARGIN + PAD
xs = [STRIP_MARGIN + PAD + i * (PANEL_W + GAP) for i in range(n)]

parts = [f'<rect x="0" y="0" width="{W}" height="{H}" fill="#ffffff" />',
         f'<rect x="{STRIP_MARGIN}" y="{STRIP_MARGIN}" width="{band_w}" '
         f'height="{PAD * 2 + PANEL_H}" rx="18" fill="#f5f8ff" '
         f'stroke="#dbe6fb" stroke-width="1" />']
parts += [harrow(xs[i] + PANEL_W + 10, xs[i + 1] - 10, y + PANEL_H / 2) for i in range(n - 1)]
parts += [f'<g transform="translate({x},{y})">\n{body}\n  </g>'
          for x, body in zip(xs, bodies)]

(PANELS / f"{PREFIX}_flow.svg").write_text(
    f'<?xml version="1.0" encoding="UTF-8" standalone="no"?>\n'
    f'<!-- Generated by {PREFIX}.ipynb - edit that notebook, not this file. -->\n'
    f'<svg width="{W}" height="{H}" viewBox="0 0 {W} {H}" version="1.1"\n'
    f'     xmlns="http://www.w3.org/2000/svg">\n'
    f'  <style>{STRIP_STYLE}  </style>\n' + "\n  ".join(parts) + "\n</svg>\n")

print(f"  {n} step SVGs + {PREFIX}_flow.svg ({W}x{H})")
wrote(PANELS / f"{PREFIX}_flow.svg")


## Panels e, f and g - true vs predicted count

Each session stores the counts-comparison figure the app drew; its first trace is the held-out
scatter, so these are the numbers the app reported. Agg is forced - otherwise matplotlib picks the
macOS backend, applies 2x Retina scaling and snaps figure sizes to whole device pixels, shifting a
panel by a couple of pixels.


In [ ]:
import json
import zipfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CS1 = REPO_ROOT / "case_study_1"
ECOLI_ZIP = CS1 / "bright_ecoli" / "mycol_saved_session_ecoli_cfu.zip"
AUREUS_ZIP = CS1 / "s_aureus" / "mycol_saved_session_s_aureus.zip"
MCOUNT_ZIP = CS1 / "mcount" / "mycol_saved_session_mcount50.zip"
MCOUNT_CSV = ASSETS / "cellpose_count_evaluation.csv"

for p in (ECOLI_ZIP, AUREUS_ZIP, MCOUNT_ZIP, MCOUNT_CSV):
    shown = p.relative_to(REPO_ROOT) if p.is_absolute() else p
    print(f"{'ok     ' if p.exists() else 'MISSING'}  {shown}")


In [ ]:
TUNED_JSON = "cellpose_tuned_counts_comparison.json"     # the fine-tuned model
BASE_JSON = "cellpose_original_counts_comparison.json"   # base cyto2, for the comparison below


def counts_from_session(zip_path, member=TUNED_JSON):
    """A stored counts-comparison figure -> DataFrame(image, true_count, pred_count).

    The figure's first trace is the scatter of held-out images: x = ground-truth count
    from the drawn masks, y = the model's count, text = "Image number: N<br>Image name: <file>".
    The second trace is the y = x line.
    """
    with zipfile.ZipFile(zip_path) as z:
        trace = json.loads(z.read(member))["data"][0]
    names = [t.split("Image name: ")[-1] for t in trace.get("text", [])]
    return pd.DataFrame({"image": names or [f"image {i}" for i in range(len(trace["x"]))],
                         "true_count": np.asarray(trace["x"], float),
                         "pred_count": np.asarray(trace["y"], float)})


def annotated_images_in(zip_path):
    """Stems of the annotated images a session ships - its full training pool."""
    with zipfile.ZipFile(zip_path) as z:
        return {Path(n).name[: -len("_masks.tif")] for n in z.namelist()
                if n.startswith("masks/") and n.endswith("_masks.tif")}


def mape(true, pred):
    """Mean absolute percentage error, the metric used throughout case study 1."""
    true, pred = np.asarray(true, float), np.asarray(pred, float)
    return float((np.abs(pred - true) / true * 100).mean())


In [ ]:
aureus_test = counts_from_session(AUREUS_ZIP)
ecoli_test = counts_from_session(ECOLI_ZIP)

for label, zip_path, test in [("s_aureus", AUREUS_ZIP, aureus_test),
                              ("bright_ecoli", ECOLI_ZIP, ecoli_test)]:
    print(f"{label:13s} {len(test):3d} held-out test images out of "
          f"{len(annotated_images_in(zip_path))} annotated "
          f"(true counts {test['true_count'].min():.0f}-{test['true_count'].max():.0f})")


In [ ]:
# --- mCount: comparable wells, minus the wells the model was fine-tuned on ---
mcount_all = pd.read_csv(MCOUNT_CSV)
mcount_train = annotated_images_in(MCOUNT_ZIP)          # 50 wells from plate image_1

stem = mcount_all["image"].str.replace(r"\.jpg$", "", regex=True)
comparable = (mcount_all["true_count"] > 0) & ~mcount_all["has_invalid_segment"]
is_train = stem.isin(mcount_train)

mcount_test = (mcount_all[comparable & ~is_train]
               .rename(columns={"model_count": "pred_count"})
               [["image", "plate", "true_count", "pred_count", "mcount_count", "nice_count"]]
               .reset_index(drop=True))

print(f"wells in CSV            : {len(mcount_all)}")
print(f"comparable wells        : {int(comparable.sum())}   (no -1 segment, true count > 0)")
print(f"  of which fine-tuned on: {int((comparable & is_train).sum())}  -> removed")
print(f"mCount test set         : {len(mcount_test)} wells across "
      f"{mcount_test['plate'].nunique()} plates")


In [ ]:
# --- the three panels, each sitting under its own plate photograph ---
CASES = [
    {"key": "s_aureus", "panel": "e", "df": aureus_test,
     "color": "#D55E00", "s": 46, "alpha": 0.75, "edge": True},      # Okabe-Ito vermillion
    {"key": "bright_ecoli", "panel": "f", "df": ecoli_test,
     "color": "#0072B2", "s": 34, "alpha": 0.60, "edge": True},      # Okabe-Ito blue
    {"key": "mcount", "panel": "g", "df": mcount_test,
     "color": "#009E73", "s": 16, "alpha": 0.40, "edge": False},     # Okabe-Ito green
]

# One type size for every label in the finished figure. A 4.5 in panel drawn at 300 dpi
# is scaled into the assembly cell's 565.33 px column, so FONT_PT = 11.2 lands on the
# 19.5 px the workflow strip's titles are rewritten to.
PANEL_SIZE_IN, DPI, FONT_PT = 4.5, 300, 11.2
plt.rcParams.update({"font.size": FONT_PT, "axes.labelsize": FONT_PT,
                     "xtick.labelsize": FONT_PT, "ytick.labelsize": FONT_PT})


def true_vs_pred(ax, true, pred, color, s, alpha, edge):
    true, pred = np.asarray(true, float), np.asarray(pred, float)
    lim = max(true.max(), pred.max()) * 1.05

    ax.plot([0, lim], [0, lim], color="#8a8a8a", lw=1.2, ls="--", zorder=1)   # perfect counter
    ax.scatter(true, pred, s=s, alpha=alpha, color=color, zorder=2,
               edgecolors="white" if edge else "none", linewidths=0.5 if edge else 0)

    ax.set_xlim(-lim * 0.02, lim)
    ax.set_ylim(-lim * 0.02, lim)
    ax.set_aspect("equal")
    ax.text(0.05, 0.95, f"MAPE {mape(true, pred):.2f}%\nn = {len(true):,}",
            transform=ax.transAxes, va="top", ha="left", fontsize=FONT_PT,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#dddddd"))
    ax.text(lim * 0.97, lim * 0.9, "y = x", color="#8a8a8a", fontsize=FONT_PT, rotation=45,
            rotation_mode="anchor", va="bottom", ha="right")
    ax.set_xlabel("True Count", fontsize=FONT_PT)
    ax.set_ylabel("Predicted Count", fontsize=FONT_PT)
    ax.grid(True, lw=0.4, color="#ededed")
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)


In [ ]:
# All three are drawn first, then given one shared axes rectangle - the tightest box
# that fits every panel's labels, squared off so aspect="equal" fills it. Left to
# constrained layout, mcount's y axis would sit further left than the other two (its
# tick labels are "70", not "700"), and the figure aligns each plate photo to this
# rectangle, so a per-panel axis would leave the photos different widths.
panels = []
for case in CASES:
    fig, ax = plt.subplots(figsize=(PANEL_SIZE_IN, PANEL_SIZE_IN), layout="constrained")
    true_vs_pred(ax, case["df"]["true_count"], case["df"]["pred_count"],
                 case["color"], case["s"], case["alpha"], case["edge"])
    fig.canvas.draw()               # constrained layout only settles on a draw
    panels.append((case, fig, ax))

boxes = [ax.get_position() for _, _, ax in panels]
x0, x1 = max(b.x0 for b in boxes), min(b.x1 for b in boxes)
y0, y1 = max(b.y0 for b in boxes), min(b.y1 for b in boxes)
side = min(x1 - x0, y1 - y0)
AXES_RECT = [x0, y0, side, side]

for case, fig, ax in panels:
    fig.set_layout_engine("none")   # else the engine re-runs on save and undoes this
    ax.set_position(AXES_RECT)
    stem = PANELS / f"scatter_{case['key']}"
    fig.savefig(f"{stem}.png", dpi=DPI, facecolor="white")
    fig.savefig(f"{stem}.svg", facecolor="white")
    d = case["df"]
    print(f"panel {case['panel']}  {case['key']:13s} -> {stem.name}.png / .svg   "
          f"(MAPE {mape(d['true_count'], d['pred_count']):.2f}%, n = {len(d):,})")

print(f"\nshared plot area  x0={AXES_RECT[0]:.4f} width={side:.4f} of the panel")


### The numbers behind the panels

What fine-tuning bought on the two plate sessions, and how the mCount panel compares against the
other two tools scored on the same wells.


In [ ]:
fine_tuning = pd.DataFrame([
    {"case": case["key"],
     "test images": len(case["df"]),
     "MAPE, base cyto2 (%)": mape(*counts_from_session(zip_path, BASE_JSON)
                                  [["true_count", "pred_count"]].to_numpy().T),
     "MAPE, fine-tuned (%)": mape(case["df"]["true_count"], case["df"]["pred_count"])}
    for case, zip_path in zip(CASES[:2], [AUREUS_ZIP, ECOLI_ZIP])
]).set_index("case").round(2)
print(fine_tuning.to_string())

tools = pd.DataFrame(
    {"MAPE (%)": [mape(mcount_test["true_count"], mcount_test[c])
                  for c in ("pred_count", "mcount_count", "nice_count")]},
    index=["Cellpose (mycol, fine-tuned)", "MCount (best grid d_0.5, lambda=38)", "NICE"],
).round(2)
print(f"\nwhole-image count error on the {len(mcount_test)} held-out mCount wells")
print(tools.to_string())


## Assembly

Lays panel **a** and the panels out on a 1788 px canvas, draws the panel letters (here and nowhere
else), and embeds the rasters as data URIs so the `.svg` stands alone. The `.svg` is the master; the
`.png` the manuscript embeds is rendered from it at 300 dpi.

> No E. coli photograph exists, so panel **c** uses `ecoli_46.tif` from the session.


In [ ]:
import base64
import io

OUT = OUTPUT / "Figure_2.svg"

FIG_W = 1788                # matches the workflow strip's native width
MARGIN, GAP_X, ROW_GAP = 20, 26, 26
LETTER_H, LETTER_SIZE = 36, 30      # strip above each row for its panel letter
TEXT_PX = 19.5              # every label in the finished figure renders at this size
CONTENT_W = FIG_W - 2 * MARGIN
COL_W = (CONTENT_W - 2 * GAP_X) / 3

FLOW_SVG = PANELS / f"{PREFIX}_flow.svg"
FLOW_MARGIN = 16            # the flow's own gutter, backed out so its band spans the figure
DARK_LEVEL = 40             # pixels at or below this count as letterbox black

PHOTOS = [
    {"letter": "b", "name": "S. aureus", "src": ASSETS / "saureus_example_image.jpg"},
    {"letter": "c", "name": "E. coli", "src": REPO_ROOT / "case_study_1" / "bright_ecoli"
     / "mycol_saved_session_ecoli_cfu" / "images" / "ecoli_46.tif"},
    {"letter": "d", "name": "mCount well", "src": ASSETS / "mcount_example_image.jpg"},
]
SCATTERS = [{"letter": "e", "src": PANELS / "scatter_s_aureus.png"},
            {"letter": "f", "src": PANELS / "scatter_bright_ecoli.png"},
            {"letter": "g", "src": PANELS / "scatter_mcount.png"}]

STYLE = """
    text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
           Helvetica, Arial, sans-serif; fill:#0f172a; }
    .fig-letter { font-size:%dpx; font-weight:700; }
""" % LETTER_SIZE


def trim_dark_border(im):
    """Drop the near-black letterbox bars some plate photographs carry.

    Everything above DARK_LEVEL is picture; getbbox then returns the tightest box
    holding it. Returns the crop and how many px came off each side.
    """
    box = im.convert("L").point(lambda p: 255 if p > DARK_LEVEL else 0).getbbox()
    if box is None:                                   # entirely dark: leave it alone
        return im, (0, 0, 0, 0)
    l, t, r, b = box
    return im.crop(box), (l, t, im.width - r, im.height - b)


def square_crop(im, size):
    """Centre-crop to a square, then scale to `size` px. Returns the crop and its native side."""
    im = im.convert("RGB")
    side = min(im.size)
    left, top = (im.width - side) // 2, (im.height - side) // 2
    im = im.crop((left, top, left + side, top + side))
    return im.resize((size, size), Image.LANCZOS), side


def data_uri(im, fmt="PNG"):
    buf = io.BytesIO()
    im.save(buf, fmt, **({"quality": 92, "optimize": True} if fmt == "JPEG" else {"optimize": True}))
    return f"data:image/{fmt.lower()};base64,{base64.b64encode(buf.getvalue()).decode()}"


def inline_flow(x, y, scale):
    """Drop the workflow SVG in as vector, keeping its own <style>.

    `x`/`y` place the flow's *band*, not its canvas: FLOW_MARGIN is backed out so the
    band's own edges land where the caller asks. The flow's 13 px titles would land at
    13*scale px, so they are rewritten to hit TEXT_PX exactly; only this inlined copy
    is touched, and the standalone flow SVG keeps its own sizing.
    """
    inner = re.sub(r"^.*?<svg[^>]*>", "", FLOW_SVG.read_text(), flags=re.S).rsplit("</svg>", 1)[0]
    # drop the flow's white backdrop: pulled flush to the figure edge it would reach up
    # over the panel letter and hide it
    inner = re.sub(r'<rect x="0" y="0" width="\d+" height="\d+" fill="#ffffff" ?/>',
                   "", inner, count=1)
    inner = re.sub(r"(\.title\s*\{[^}]*?font-size:)[\d.]+px",
                   lambda m: f"{m.group(1)}{TEXT_PX / scale:.2f}px", inner, count=1)
    ox, oy = x - FLOW_MARGIN * scale, y - FLOW_MARGIN * scale
    return f'<g transform="translate({ox:.2f},{oy:.2f}) scale({scale:.5f})">{inner}</g>'


In [ ]:
cell = int(round(COL_W))

# the scatters set the row height; the photos above them are squared off to the width
# of the scatter's plot area, so each picture starts on its panel's y axis and ends at
# the far end of its x axis
scatter_ims = [Image.open(s["src"]) for s in SCATTERS]
scatter_h = int(round(cell * max(im.height / im.width for im in scatter_ims)))
photo_inset = AXES_RECT[0] * cell
photo_w = int(round(AXES_RECT[2] * cell))
print(f"figure width {FIG_W}, column {cell} px")
print(f"  photo align    plot area x0={AXES_RECT[0]:.4f} w={AXES_RECT[2]:.4f} -> "
      f"inset {photo_inset:.1f} px, {photo_w} px wide")

photo_uris = []
for spec in PHOTOS:
    im = Image.open(spec["src"])
    raw = im.size
    im, trimmed = trim_dark_border(im)
    square, native = square_crop(im, photo_w)
    photo_uris.append(data_uri(square, "JPEG"))
    print(f"  panel {spec['letter']}  {spec['name']:<12} {spec['src'].name:<20} "
          f"{raw[0]}x{raw[1]} -> {photo_w}x{photo_w} from {native} px native"
          + (f", trimmed l/t/r/b {trimmed}" if any(trimmed) else ""))

# The flow spans the pictures, not the figure: the photos are inset onto the scatter's
# y axis, so the band runs from the first photo's left edge to the last photo's right.
flow_x = MARGIN + photo_inset
flow_band_w = (MARGIN + 2 * (COL_W + GAP_X) + photo_inset + photo_w) - flow_x
flow_w, flow_canvas_h = (float(v) for v in
                         re.search(r'viewBox="0 0 ([\d.]+) ([\d.]+)"',
                                   FLOW_SVG.read_text()[:600]).groups())
flow_scale = flow_band_w / (flow_w - 2 * FLOW_MARGIN)
flow_h = (flow_canvas_h - 2 * FLOW_MARGIN) * flow_scale
print(f"  panel a  workflow     {FLOW_SVG.name:<20} {flow_w:.0f}x{flow_canvas_h:.0f} -> "
      f"band {flow_band_w:.0f} px ({flow_scale:.2f}x) at x={flow_x:.1f}")

y_a = MARGIN + LETTER_H
y_b = y_a + flow_h + ROW_GAP + LETTER_H
y_c = y_b + photo_w + ROW_GAP + LETTER_H
fig_h = int(round(y_c + scatter_h + MARGIN))

parts = [f'<rect x="0" y="0" width="{FIG_W}" height="{fig_h}" fill="#ffffff" />',
         f'<text class="fig-letter" x="{MARGIN}" y="{MARGIN + LETTER_SIZE}">a</text>',
         inline_flow(flow_x, y_a, flow_scale)]

for i, (spec, uri) in enumerate(zip(PHOTOS, photo_uris)):
    x = MARGIN + i * (COL_W + GAP_X)
    # the letter stays on the column edge; only the picture is inset onto the y axis
    parts.append(f'<text class="fig-letter" x="{x:.1f}" y="{y_b - LETTER_H + LETTER_SIZE:.1f}">'
                 f'{spec["letter"]}</text>')
    parts.append(f'<image x="{x + photo_inset:.1f}" y="{y_b:.1f}" width="{photo_w}" '
                 f'height="{photo_w}" href="{uri}" />')

for i, (spec, im) in enumerate(zip(SCATTERS, scatter_ims)):
    x = MARGIN + i * (COL_W + GAP_X)
    parts.append(f'<text class="fig-letter" x="{x:.1f}" y="{y_c - LETTER_H + LETTER_SIZE:.1f}">'
                 f'{spec["letter"]}</text>')
    parts.append(f'<image x="{x:.1f}" y="{y_c:.1f}" width="{cell}" '
                 f'height="{int(round(cell * im.height / im.width))}" href="{data_uri(im)}" />')

OUT.write_text(
    f'<?xml version="1.0" encoding="UTF-8" standalone="no"?>\n'
    f'<!-- Generated by {PREFIX}.ipynb - edit that notebook, not this file. -->\n'
    f'<svg width="{FIG_W}" height="{fig_h}" viewBox="0 0 {FIG_W} {fig_h}"\n'
    f'     version="1.1" xmlns="http://www.w3.org/2000/svg"\n'
    f'     xmlns:xlink="http://www.w3.org/1999/xlink">\n'
    f'  <title>Figure 2 - CFU counting</title>\n'
    f'  <style>{STYLE}  </style>\n  ' + "\n  ".join(parts) + "\n</svg>\n")

print()
wrote(OUT)
rasterise(OUT, OUTPUT / "Figure_2.png")


## What this notebook wrote


In [ ]:
from IPython.display import display

for p in sorted(OUTPUT.rglob("*")):
    if p.is_file():
        print(f"  {str(p.relative_to(OUTPUT)):44} {p.stat().st_size:>9,} B")

im = Image.open(OUTPUT / "Figure_2.png")
print(f"\nFigure 2   {im.width} x {im.height} px")
im.thumbnail((760, 760))
display(im.convert("RGB"))
